# PydanticAI together with Pydantic Models

In [2]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat format
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

from dotenv import load_dotenv  # reads key=value pairs from a .env file

# Load the variables from your .env file (e.g. NVIDIA_API=...) into the environment.
# This keeps secret keys out of your code so you don't accidentally commit them to git.
load_dotenv()

# Nvidia's API uses the same "chat completion" format as OpenAI,
# so we can reuse the OpenAI classes to talk to Nvidia models.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",
    provider=OpenAIProvider(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ["NVIDIA_API"],
    ),
)


In [5]:
employee_simulator = Agent(
    model=model,
    instructions="""Du är expert på att simulera anställda i Sverige.
    Följ detta format:
    name: "Bertil"
    age: 40
    salary: 35000
    role: "IT-supporter"
    ort: "Göteborg"
    """,
)

result = await employee_simulator.run(
    "Ge mig en data engineer som jobbar i Stockholm"
)
print(result.output)

employee_simulator = Agent(
    model=model,
    instructions="""Du är expert på att simulera anställda i Sverige.
    Följ detta format:
    name: "Bertil"
    age: 40
    salary: 35000
    role: "IT-supporter"
    ort: "Göteborg"
    """,
)

result = await employee_simulator.run("Ge mig en data engineer som jobbar i stockholm ")
print(result.output)

name: "Johan"
age: 34
salary: 58000
role: "Data engineer"
ort: "Stockholm"
name: "David Johansson"
age: 35
salary: 48000
role: "Data engineer"
ort: "Stockholm"


## Force a structured output using pydantic

In [6]:
from pydantic import BaseModel, Field


class EmployeeModel(BaseModel):
    name: str = Field(
        description="för och efternamn, inte jättelångt, svenska eller internationella namn"
    )
    age: int = Field(description="ålder från 18 till 67 år", ge=18, le=67)
    salary: int = Field(
        description="lön mellan 30k-60k, ju mer senior position, desto högre lön",
        ge=30_000,
        lt=60_000,
    )
    role: str = Field(description="yrke/roll")
    description: str = Field(description="kort beskrivning, 2 meningar på rollen")


employee1 = EmployeeModel(
    name="Bertil",
    age=40,
    salary=30000,
    role="IT supporter",
    description="Hjälper folk med IT",
)

employee1

EmployeeModel(name='Bertil', age=40, salary=30000, role='IT supporter', description='Hjälper folk med IT')

In [8]:
employee_simulator = Agent(
    model=model,
    instructions="Du är expert på att simulera anställda i Sverige",
)

result = await employee_simulator.run("Ge mig en data engineer i stockholm")

result.output

'Jag kan inte erbjuda kontaktuppgifter till riktiga individer eller åtkomst till privatpersoners data. Däremot kan jag hjälpa dig på annat sätt om du söker en dataingenjör i Stockholm:\n\n🧑\u200d💻 **Exempel på en fiktiv profil (för inspiration)**\nNamn: Emma Johansson  \nRoll: Data Engineer  \nErfarenhet: 4 år inom data och analytics  \nArbetsort: Stockholm (hybrid/kontor)  \nTeknisk stack: Python, SQL, Apache Airflow, dbt, AWS (S3, Redshift, Glue), Apache Kafka, Snowflake, Power BI  \nBakgrund: tidigare på ett finansbolag där hon byggde ETL-pipelines och datamodeller för affärsdata. Just nu söker hon en roll där hon får arbeta med realtidsdata och modern cloud-infrastruktur.\n\n🔧 **Vanliga färdigheter och verktyg hos dataingenjörer i Stockholm**\n- Programmering: Python, Scala, SQL\n- Cloud: AWS, GCP eller Azure (många Stockholm-företag är tunga AWS-användare)\n- Pipeline-verktyg: Airflow, Prefect, dbt, Kafka, Spark\n- Datalager: Snowflake, Redshift, BigQuery, Postgres\n- Övriga: Vers

In [9]:
result = await employee_simulator.run(
    "Ge mig en data engineer i stockholm", output_type=EmployeeModel
)
result

AgentRunResult(output=EmployeeModel(name='Johan Svensson', age=32, salary=45000, role='Data Engineer', description='Designs and maintains scalable data pipelines and infrastructure for large-scale data processing. Collaborates with cross-functional teams to ensure data quality, reliability, and accessibility.'))

In [10]:
employee2 = result.output
employee2

EmployeeModel(name='Johan Svensson', age=32, salary=45000, role='Data Engineer', description='Designs and maintains scalable data pipelines and infrastructure for large-scale data processing. Collaborates with cross-functional teams to ensure data quality, reliability, and accessibility.')

In [11]:
type(employee2)

__main__.EmployeeModel

In [12]:
employee2.name + " är cool"

'Johan Svensson är cool'

In [13]:
employee2.salary

45000

In [14]:
employee2.__dict__.keys()

dict_keys(['name', 'age', 'salary', 'role', 'description'])

In [15]:
employee2.__dict__.values()

dict_values(['Johan Svensson', 32, 45000, 'Data Engineer', 'Designs and maintains scalable data pipelines and infrastructure for large-scale data processing. Collaborates with cross-functional teams to ensure data quality, reliability, and accessibility.'])

In [16]:
for key, value in employee2.__dict__.items():
    print(f"{key}: {value}")

name: Johan Svensson
age: 32
salary: 45000
role: Data Engineer
description: Designs and maintains scalable data pipelines and infrastructure for large-scale data processing. Collaborates with cross-functional teams to ensure data quality, reliability, and accessibility.


## Simulate many employees

In [17]:
result = await employee_simulator.run("Ge mig totalt 10 anställda inom IT/HR/marketing",
                                      output_type=list[EmployeeModel])
result

AgentRunResult(output=[EmployeeModel(name='Anna Lindberg', age=30, salary=45000, role='Software Developer', description='Develops and maintains enterprise applications. Works closely with cross-functional teams to deliver robust software solutions.'), EmployeeModel(name='Erik Johansson', age=35, salary=55000, role='Senior DevOps Engineer', description='Manages CI/CD pipelines and cloud infrastructure to ensure seamless deployments. Optimizes system performance and reliability.'), EmployeeModel(name='Sara Nilsson', age=27, salary=40000, role='UI/UX Designer', description='Designs intuitive user interfaces and conducts user research. Creates visually appealing and accessible digital experiences.'), EmployeeModel(name='Lena Karlsson', age=42, salary=48000, role='HR Business Partner', description='Supports employee relations and talent acquisition initiatives. Advises managers on HR policies and compliance.'), EmployeeModel(name='Mikael Persson', age=48, salary=52000, role='Recruitment Spe

## Task - calculate mean salary 

In [18]:
employees = result.output

employees

[EmployeeModel(name='Anna Lindberg', age=30, salary=45000, role='Software Developer', description='Develops and maintains enterprise applications. Works closely with cross-functional teams to deliver robust software solutions.'),
 EmployeeModel(name='Erik Johansson', age=35, salary=55000, role='Senior DevOps Engineer', description='Manages CI/CD pipelines and cloud infrastructure to ensure seamless deployments. Optimizes system performance and reliability.'),
 EmployeeModel(name='Sara Nilsson', age=27, salary=40000, role='UI/UX Designer', description='Designs intuitive user interfaces and conducts user research. Creates visually appealing and accessible digital experiences.'),
 EmployeeModel(name='Lena Karlsson', age=42, salary=48000, role='HR Business Partner', description='Supports employee relations and talent acquisition initiatives. Advises managers on HR policies and compliance.'),
 EmployeeModel(name='Mikael Persson', age=48, salary=52000, role='Recruitment Specialist', descript

In [19]:
salary = 0

for employee in employees:
    salary += employee.salary

salary/len(employees)

48000.0

## Convert to dataframe

In [20]:
employees[0].model_dump()

{'name': 'Anna Lindberg',
 'age': 30,
 'salary': 45000,
 'role': 'Software Developer',
 'description': 'Develops and maintains enterprise applications. Works closely with cross-functional teams to deliver robust software solutions.'}

In [21]:
import pandas as pd
df = pd.DataFrame([employee.model_dump() for employee in employees])
df.head()

,name,age,salary,role,description
0,Anna Lindberg,30,45000,Software Developer,Develops and maintains enterprise applications...
1,Erik Johansson,35,55000,Senior DevOps Engineer,Manages CI/CD pipelines and cloud infrastructu...
2,Sara Nilsson,27,40000,UI/UX Designer,Designs intuitive user interfaces and conducts...
3,Lena Karlsson,42,48000,HR Business Partner,Supports employee relations and talent acquisi...
4,Mikael Persson,48,52000,Recruitment Specialist,Identifies and attracts top talent through var...


In [22]:
df.to_csv("employees.csv", index=None)